# Video 3 run-through: sim_pushTV2

Headings match the Parts of `Week 3 Script - I Generated 1000 Robot Episodes...` in the vault.

**Rules for the take**
- One kernel, top to bottom. If it dies, restart and run from the top of the current Part.
- Cells marked **PRE-RUN** were run before recording; show their output, do not re-run.
- Cells marked **LIVE** run on camera.

## Setup

In [ ]:
import json
import subprocess
import time
from pathlib import Path

import gymnasium as gym
import gym_pusht
import numpy as np
import torch

WEEK = Path("/Users/snudurupati/Projects/physical-ai-data-architect/weeks/week-03")
MIG  = str(WEEK / "model" / "diffusion_pusht_migrated")
SIM  = str(WEEK / "data" / "pusht_sim")
EVAL_DIR = WEEK / "outputs" / "dp_official_eval" / "videos" / "pusht_0"

## Part 3: the environment — LIVE

In [ ]:
env = gym.make("gym_pusht/PushT-v0", obs_type="state", render_mode="rgb_array")
obs, info = env.reset(seed=0)
print("obs (pusher_x, pusher_y, block_x, block_y, block_angle):")
print(obs)
print("goal_pose:", info["goal_pose"])

In [ ]:
print("action space low / high (absolute target pixels):")
print(env.action_space.low, env.action_space.high)

Facts spoken here (from `pusht.py`): success = coverage > 0.95 on the block's center of mass; 300-step cap; no gravity, no damping.

## Part 4: random policy — LIVE (100 episodes take seconds)

In [ ]:
def run_policy(make_action, n_episodes=100):
    successes, fail_lengths, max_covs = 0, [], []
    for ep in range(n_episodes):
        obs, info = env.reset(seed=ep)
        done, n, best = False, 0, 0.0
        while not done and n < 300:
            obs, r, term, trunc, info = env.step(make_action(obs, info))
            best = max(best, info["coverage"])
            done = term or trunc
            n += 1
        successes += int(term)
        if not term:
            fail_lengths.append(n)
        max_covs.append(best)
    print(f"{successes}/{n_episodes} successes, "
          f"mean failure length {np.mean(fail_lengths):.0f}, "
          f"mean max coverage {np.mean(max_covs):.3f}")

run_policy(lambda obs, info: env.action_space.sample().astype(np.float32))

## Part 5: the star policy — show code + output (PRE-RUN ok, 100 episodes run in seconds)

In [ ]:
def star_policy(margin):
    def policy(obs, info):
        goal = np.array([info["goal_pose"][0], info["goal_pose"][1]])
        block = np.array([obs[2], obs[3]])
        d = block - goal
        norm = np.linalg.norm(d)
        u = d / norm if norm > 0 else np.array([0.0, 0.0])
        return (block + u * margin).astype(np.float32)
    return policy

run_policy(star_policy(margin=40))

In [ ]:
run_policy(star_policy(margin=10))   # the "cowboy": spins the T

Instrumented trace for the margin-40 story (LIVE, one episode, prints 6 rows):

In [ ]:
obs, info = env.reset(seed=0)
print("step | pusher (x, y) | action (x, y) | block (x, y) | coverage")
for step in range(300):
    action = star_policy(margin=40)(obs, info)
    obs, r, term, trunc, info = env.step(action)
    if step % 50 == 0:
        print(f"{step:4d} | {obs[0]:6.1f} {obs[1]:6.1f} | "
              f"{action[0]:6.1f} {action[1]:6.1f} | "
              f"{obs[2]:6.1f} {obs[3]:6.1f} | {info['coverage']:.3f}")

Diagram on screen here: `../assets/policy_v1.png`

## Part 6 (optional): the LLM policy — show code + the two bugs; run only if you rehearsed it

## Part 7: the checkpoint and the official eval — show, do not re-run (95 s/episode)

In [ ]:
!ls -lh {WEEK}/model/diffusion_pusht/ | tail -6

Spoken here, from the captured notebook output: the `Unexpected key(s) ... normalize_inputs` warning, and the eval's first error `Config file 'policy_preprocessor.json' not found`, fixed by:

```
uv run python -m lerobot.processor.migrate_policy_normalization \
  --pretrained-path weeks/week-03/model/diffusion_pusht \
  --output-dir weeks/week-03/model/diffusion_pusht_migrated
```

Official eval command (shown, pre-run; its console output is the saved `pc_success: 60.0` block):

In [ ]:
!uv run python -m lerobot.scripts.lerobot_eval \
  --policy.path=weeks/week-03/model/diffusion_pusht_migrated \
  --env.type=pusht --eval.n_episodes=5 --eval.batch_size=1 \
  --policy.device=cpu --output_dir=weeks/week-03/outputs/dp_official_eval

Success clip (LIVE, plays in the system player — mute it first):

In [ ]:
subprocess.run(["open", str(EVAL_DIR / "eval_episode_1.mp4")])

## Part 8: the dataset writer — explain the cells, PRE-RUN, do not execute

Already run before recording: `RANDOM 1000 done in 436s`, `DIFFUSION 50 done in 5309s` (88 min 30 s, ~106 s/episode). Show these two log lines from the terminal.

In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from lerobot.policies.diffusion.modeling_diffusion import DiffusionPolicy
from lerobot.policies.factory import make_pre_post_processors
from lerobot.envs.utils import preprocess_observation

features = {
    "observation.image": {"dtype": "image",   "shape": (96, 96, 3), "names": ["height", "width", "channel"]},
    "observation.state": {"dtype": "float32", "shape": (2,), "names": ["x", "y"]},
    "action":            {"dtype": "float32", "shape": (2,), "names": ["x", "y"]},
    "is_success":        {"dtype": "bool",    "shape": (1,), "names": None},
}
print(json.dumps(features, indent=1))

The `rollout(make_action, n_episodes, tag)` function and the two `make_action`s (random lambda, diffusion function) are the same cells as in `sim_pushT.ipynb` — scroll there for the code on camera.

## Pre-run check: prints the numbers the script quotes — run BEFORE the take

Measured run of this cell: 1,050 episodes, 310,759 frames; random 1,000 episodes / 300,000 frames / 0 successes; diffusion 50 episodes / 10,759 frames / 36 successes.

In [ ]:
info = json.loads((Path(SIM) / "meta" / "info.json").read_text())
print("total_episodes:", info["total_episodes"])
print("total_frames:  ", info["total_frames"])
print("fps:           ", info["fps"])
print("robot_type:    ", info["robot_type"])

# Per-task counts straight from the parquet files — no Spark session needed here.
import glob
import pyarrow.parquet as pq
from collections import defaultdict

task_names = {r["task_index"]: r["__index_level_0__"]
              for r in pq.read_table(Path(SIM) / "meta" / "tasks.parquet").to_pylist()}
eps, frames, successes = defaultdict(set), defaultdict(int), defaultdict(set)
for f in glob.glob(str(Path(SIM) / "data" / "*" / "*.parquet")):
    t = pq.read_table(f, columns=["task_index", "episode_index", "is_success"])
    for ti, ep, ok in zip(t.column("task_index").to_pylist(),
                          t.column("episode_index").to_pylist(),
                          t.column("is_success").to_pylist()):
        eps[ti].add(ep); frames[ti] += 1
        if ok: successes[ti].add(ep)
for ti in sorted(eps):
    print(f"{task_names[ti]:>14}: {len(eps[ti]):4d} episodes, {frames[ti]:6d} frames, "
          f"{len(successes[ti]):3d} success episodes")

These are the script's numbers: 0 of 1,000 random-policy episodes true, 36 of 50 diffusion episodes true (72%). Point `01_profiler.ipynb`'s `REPO_ID`/`ROOT` at `SIM` next; it writes `results/pusht_sim.json` for Parts 9–10 of the script.